In [0]:
# ============================================================
# Silver — Source 12: Reviews and Support Tickets
#
# Two tables: reviews and tickets
#
# Reviews transformations:
#   - Cast created_at to timestamp
#   - Validate rating 1-5 (null rating → quarantine)
#   - Reject null record_id, order_id, product_sku → quarantine
#
# Tickets transformations:
#   - Cast created_at, resolved_at to timestamp
#   - Normalise status, priority, category, channel
#   - Reject null record_id, order_id → quarantine
#
# Source:  bronze.src_12_reviews.*
# Target:  silver.src_12_reviews.*
# Quarantine: silver.quarantine.src_12_reviews
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'

VALID_TICKET_STATUSES = ['open', 'pending', 'resolved', 'closed', 'cancelled']
VALID_PRIORITIES = ['low', 'medium', 'high', 'urgent']

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_12_reviews')
print('Silver Source 12 Reviews + Tickets — starting...')


In [0]:
# ── REVIEWS ───────────────────────────────────────────────────
print('\n--- REVIEWS ---')
bronze_reviews = spark.table(f'{BRONZE_CATALOG}.src_12_reviews.reviews')
total = bronze_reviews.count()

df_reviews = bronze_reviews \
    .withColumn('created_at', F.to_timestamp(F.col('created_at')))

bad_reviews = df_reviews.filter(
    F.col('record_id').isNull() |
    F.col('order_id').isNull() |
    F.col('product_sku').isNull() |
    F.col('rating').isNull() |
    (F.col('rating') < 1) |
    (F.col('rating') > 5)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('reviews'))

good_reviews = df_reviews.filter(
    F.col('record_id').isNotNull() &
    F.col('order_id').isNotNull() &
    F.col('product_sku').isNotNull() &
    F.col('rating').isNotNull() &
    (F.col('rating') >= 1) &
    (F.col('rating') <= 5)
).dropDuplicates(['record_id'])

bad_count = bad_reviews.count()
good_count = good_reviews.count()
print(f'Reviews: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

TARGET = f'{SILVER_CATALOG}.src_12_reviews.reviews'
if spark.catalog.tableExists(TARGET):
    dt = DeltaTable.forName(spark, TARGET)
    dt.alias('t').merge(good_reviews.alias('s'), 't.record_id = s.record_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good_reviews.write.format('delta').mode('overwrite').saveAsTable(TARGET)
print('✅ silver.src_12_reviews.reviews written')

if bad_count > 0:
    bad_reviews.select(
        F.lit('src_12_reviews').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad_reviews.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    ).write.format('delta').mode('append').option('mergeSchema','true') \
     .saveAsTable(f'{SILVER_CATALOG}.quarantine.src_12_reviews')


In [0]:
# ── TICKETS ───────────────────────────────────────────────────
print('\n--- TICKETS ---')
bronze_tickets = spark.table(f'{BRONZE_CATALOG}.src_12_reviews.tickets')
total = bronze_tickets.count()

df_tickets = bronze_tickets \
    .withColumn('created_at',  F.to_timestamp(F.col('created_at'))) \
    .withColumn('resolved_at', F.to_timestamp(F.col('resolved_at')))

# Normalise
df_tickets = df_tickets \
    .withColumn('status',   F.lower(F.trim(F.col('status')))) \
    .withColumn('priority', F.lower(F.trim(F.col('priority')))) \
    .withColumn('category', F.lower(F.trim(F.col('category')))) \
    .withColumn('channel',  F.lower(F.trim(F.col('channel'))))

bad_tickets = df_tickets.filter(
    F.col('record_id').isNull() |
    F.col('order_id').isNull() |
    F.col('customer_id').isNull() |
    ~F.col('status').isin(VALID_TICKET_STATUSES) |
    ~F.col('priority').isin(VALID_PRIORITIES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('tickets'))

good_tickets = df_tickets.filter(
    F.col('record_id').isNotNull() &
    F.col('order_id').isNotNull() &
    F.col('customer_id').isNotNull() &
    F.col('status').isin(VALID_TICKET_STATUSES) &
    F.col('priority').isin(VALID_PRIORITIES)
).dropDuplicates(['record_id'])

bad_count = bad_tickets.count()
good_count = good_tickets.count()
print(f'Tickets: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

TARGET = f'{SILVER_CATALOG}.src_12_reviews.tickets'
if spark.catalog.tableExists(TARGET):
    dt = DeltaTable.forName(spark, TARGET)
    dt.alias('t').merge(good_tickets.alias('s'), 't.record_id = s.record_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good_tickets.write.format('delta').mode('overwrite').saveAsTable(TARGET)
print('✅ silver.src_12_reviews.tickets written')

if bad_count > 0:
    bad_tickets.select(
        F.lit('src_12_reviews').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad_tickets.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    ).write.format('delta').mode('append').option('mergeSchema','true') \
     .saveAsTable(f'{SILVER_CATALOG}.quarantine.src_12_reviews')


In [0]:
# ── VERIFY ────────────────────────────────────────────────────
r_count = spark.sql(f'SELECT COUNT(*) as cnt FROM {SILVER_CATALOG}.src_12_reviews.reviews').collect()[0]['cnt']
t_count = spark.sql(f'SELECT COUNT(*) as cnt FROM {SILVER_CATALOG}.src_12_reviews.tickets').collect()[0]['cnt']
print(f'silver.src_12_reviews.reviews: {r_count} rows')
print(f'silver.src_12_reviews.tickets: {t_count} rows')
spark.sql(f'SELECT rating, COUNT(*) as cnt FROM {SILVER_CATALOG}.src_12_reviews.reviews GROUP BY rating ORDER BY rating').show()
spark.sql(f'SELECT status, priority, COUNT(*) as cnt FROM {SILVER_CATALOG}.src_12_reviews.tickets GROUP BY status, priority ORDER BY cnt DESC').show()
